In [1]:
import pandas as pd
import numpy as np
import re

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

DATA_PATH = Path("../data/cleaned/cyberguard_text_dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset:", df.shape)
print(df["cyberguard_label"].value_counts())

Dataset: (163924, 4)
cyberguard_label
1    85450
0    78474
Name: count, dtype: int64


In [2]:
def create_template_key(text):
    text = str(text).lower()

    # Remove email addresses
    text = re.sub(
        r'\b[\w\.-]+@[\w\.-]+\.\w+\b',
        ' EMAIL ',
        text
    )

    # Remove URLs
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' URL ',
        text
    )

    # Replace numbers
    text = re.sub(r'\b\d+\b', ' NUMBER ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [3]:
df["template_text"] = df["text"].apply(create_template_key)

In [4]:
df["template_group"] = (
    df["template_text"]
    .str[:500]
)

In [5]:
group_label_counts = pd.crosstab(
    df["template_group"],
    df["cyberguard_label"]
)

conflicting_groups = group_label_counts[
    (group_label_counts.get(0, 0) > 0) &
    (group_label_counts.get(1, 0) > 0)
]

print(
    "Groups containing both labels:",
    len(conflicting_groups)
)

Groups containing both labels: 0


In [6]:
print(
    "Percentage of groups with conflicting labels:",
    round(
        len(conflicting_groups) /
        df["template_group"].nunique() * 100,
        2
    ),
    "%"
)

Percentage of groups with conflicting labels: 0.0 %


In [7]:
X = df["text"]
y = df["cyberguard_label"]
groups = df["template_group"]

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    sgkf.split(X, y, groups)
)

X_train_robust = X.iloc[train_idx]
X_test_robust = X.iloc[test_idx]

y_train_robust = y.iloc[train_idx]
y_test_robust = y.iloc[test_idx]

print("Training:", len(X_train_robust))
print("Testing :", len(X_test_robust))

print("\nTraining distribution:")
print(
    y_train_robust
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting distribution:")
print(
    y_test_robust
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Training: 131139
Testing : 32785

Training distribution:
cyberguard_label
1    52.13
0    47.87
Name: proportion, dtype: float64

Testing distribution:
cyberguard_label
1    52.13
0    47.87
Name: proportion, dtype: float64


In [8]:
train_groups = set(
    df.iloc[train_idx]["template_group"]
)

test_groups = set(
    df.iloc[test_idx]["template_group"]
)

overlap_groups = train_groups.intersection(test_groups)

print(
    "Train-test template group overlap:",
    len(overlap_groups)
)

Train-test template group overlap: 0


In [9]:
tfidf_robust = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=200000
)

X_train_robust_tfidf = tfidf_robust.fit_transform(
    X_train_robust
)

X_test_robust_tfidf = tfidf_robust.transform(
    X_test_robust
)

print(
    "Training TF-IDF:",
    X_train_robust_tfidf.shape
)

print(
    "Testing TF-IDF:",
    X_test_robust_tfidf.shape
)

Training TF-IDF: (131139, 200000)
Testing TF-IDF: (32785, 200000)


In [10]:
robust_svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    random_state=42
)

robust_svm.fit(
    X_train_robust_tfidf,
    y_train_robust
)

robust_pred = robust_svm.predict(
    X_test_robust_tfidf
)

In [11]:
robust_accuracy = accuracy_score(
    y_test_robust,
    robust_pred
)

robust_precision = precision_score(
    y_test_robust,
    robust_pred
)

robust_recall = recall_score(
    y_test_robust,
    robust_pred
)

robust_f1 = f1_score(
    y_test_robust,
    robust_pred
)

print("===== Robust Template-Aware SVM =====")

print(f"Accuracy : {robust_accuracy:.6f}")
print(f"Precision: {robust_precision:.6f}")
print(f"Recall   : {robust_recall:.6f}")
print(f"F1 Score : {robust_f1:.6f}")

===== Robust Template-Aware SVM =====
Accuracy : 0.998627
Precision: 0.998188
Recall   : 0.999181
F1 Score : 0.998684


In [12]:
print(
    classification_report(
        y_test_robust,
        robust_pred,
        target_names=["Legitimate", "Suspicious"]
    )
)

              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00     15695
  Suspicious       1.00      1.00      1.00     17090

    accuracy                           1.00     32785
   macro avg       1.00      1.00      1.00     32785
weighted avg       1.00      1.00      1.00     32785



In [13]:
robust_cm = confusion_matrix(
    y_test_robust,
    robust_pred
)

print("Confusion Matrix:")
print(robust_cm)

Confusion Matrix:
[[15664    31]
 [   14 17076]]


In [14]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../models")
RESULTS_DIR = Path("../results")

MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

joblib.dump(
    robust_svm,
    MODEL_DIR / "cyberguard_text_svm.pkl"
)

joblib.dump(
    tfidf_robust,
    MODEL_DIR / "cyberguard_tfidf.pkl"
)

robust_metrics = {
    "accuracy": robust_accuracy,
    "precision": robust_precision,
    "recall": robust_recall,
    "f1": robust_f1,
    "true_negatives": robust_cm[0, 0],
    "false_positives": robust_cm[0, 1],
    "false_negatives": robust_cm[1, 0],
    "true_positives": robust_cm[1, 1]
}

pd.DataFrame([robust_metrics]).to_csv(
    RESULTS_DIR / "robust_svm_metrics.csv",
    index=False
)

print("Validated CyberGuard SVM saved successfully.")

Validated CyberGuard SVM saved successfully.
